In [49]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")


In [63]:
from langchain_community.document_loaders import PyPDFLoader

# Define the path to your PDF file
file_path = r"C:\Users\saipr\New_Idea\10th_social_em.pdf"

# Initialize the loader
loader = PyPDFLoader(file_path)

# Load the document
pages = loader.load()

In [64]:
len(pages)

346

In [66]:
# The correct import path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

# Example usage (assuming you have your 'pages' from the PyPDFLoader)
docs = text_splitter.split_documents(pages)

In [68]:
len(docs)

1044

In [69]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings

# 1. Load the variables from your .env file
load_dotenv()

# 2. Retrieve your specific token and set it to the variable LangChain expects
hf_token = os.getenv("HUGGINFACE_ENDPOIN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

# 3. Initialize the embedding model via the Inference API
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name
)

# 4. Test it out on a sample text
text_to_embed = "Hello, Hugging Face!"
embedding_result = embeddings.embed_query(text_to_embed)

print("Successfully connected to Hugging Face!")
print(f"Generated an embedding with {len(embedding_result)} dimensions.")

c:\Users\saipr\New_Idea\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully connected to Hugging Face!
Generated an embedding with 768 dimensions.


In [70]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. Load the variables from your .env file
load_dotenv()

# 2. Set up Hugging Face Token
hf_token = os.getenv("HUGGINFACE_ENDPOIN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

# 3. Initialize the embedding model via the Inference API
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name
)

# 4. Set up Pinecone API Key
pinecone_api_key = os.getenv("PINECONE_API_KEY")
os.environ["PINECONE_API_KEY"] = pinecone_api_key

# Initialize Pinecone client
pc = Pinecone(api_key=pinecone_api_key)
index_name = "social-textbook-index"

# 5. Create the Pinecone index if it doesn't already exist
# The dimension MUST be 768 because 'all-mpnet-base-v2' outputs 768-dimensional vectors
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# 6. Convert the 'docs' into embeddings and store them in Pinecone
print("Uploading documents to Pinecone... This might take a moment.")

vector_store = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

print("Successfully converted 'docs' to embeddings and stored them in Pinecone!")

Uploading documents to Pinecone... This might take a moment.


HfHubHTTPError: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/hf-inference/models/sentence-transformers/all-mpnet-base-v2/pipeline/feature-extraction' (Amz CF ID: DkRAWJ5Io-xG3d-dBaU5DXJ_aWla8l-Z02mbPaEKnssNL2xAVrsujA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504

In [71]:
import time
from langchain_pinecone import PineconeVectorStore

# ... [Steps 1 through 5 remain exactly the same] ...

# 6. Initialize the Vector Store object once
vector_store = PineconeVectorStore(
    index_name=index_name, 
    embedding=embeddings
)

print("Uploading documents to Pinecone in batches to avoid server timeouts...")

# Set a smaller batch size to prevent overloading the Hugging Face API
batch_size = 32  

# Loop through the docs in batches
for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    print(f"Processing chunks {i} to {i + len(batch)} out of {len(docs)}...")
    
    # Upload the specific batch
    vector_store.add_documents(documents=batch)
    
    # Pause for 2 seconds to let the free Hugging Face API recover
    time.sleep(2)

print("Successfully converted all 'docs' to embeddings and stored them in Pinecone!")

Uploading documents to Pinecone in batches to avoid server timeouts...
Processing chunks 0 to 32 out of 1044...
Processing chunks 32 to 64 out of 1044...
Processing chunks 64 to 96 out of 1044...
Processing chunks 96 to 128 out of 1044...
Processing chunks 128 to 160 out of 1044...
Processing chunks 160 to 192 out of 1044...
Processing chunks 192 to 224 out of 1044...
Processing chunks 224 to 256 out of 1044...
Processing chunks 256 to 288 out of 1044...
Processing chunks 288 to 320 out of 1044...
Processing chunks 320 to 352 out of 1044...
Processing chunks 352 to 384 out of 1044...
Processing chunks 384 to 416 out of 1044...
Processing chunks 416 to 448 out of 1044...
Processing chunks 448 to 480 out of 1044...
Processing chunks 480 to 512 out of 1044...
Processing chunks 512 to 544 out of 1044...
Processing chunks 544 to 576 out of 1044...
Processing chunks 576 to 608 out of 1044...
Processing chunks 608 to 640 out of 1044...
Processing chunks 640 to 672 out of 1044...
Processing ch

In [ ]:
# 1. Define your question/query
query = "The Himalayan ranges run in the west-east direction in the form of an arch with a distance of about _______"

# 2. Perform the similarity search, requesting the top 3 chunks (k=3)
top_docs = vector_store.similarity_search(query, k=3)

# 3. Print the results
print(f"Top 3 retrieved documents for the query:\n'{query}'\n")

for i, doc in enumerate(top_docs):
    print(f"--- Document {i+1} ---")
    print(doc.page_content)
    print("\n")

Top 3 retrieved documents for the query:
'The Himalayan ranges run in the west-east direction in the form of an arch with a distance of about _______'

--- Document 1 ---
5Free distribution by T.S. Government 2019-20
 Major Relief Divisions
The relief features of Indian landmass can be divided into the following groups:
1. The Himalayas 2. The Indo-Gangetic Plain
3. The Peninsular Plateau 4. The Coastal plains
5. The Desert 6. The Islands
The Himalayas
The Himalayan ranges run in the west-east direction in the form of an arch with
a distance of about 2,400 kms. Their width differs from 500 kms in the western
regions to 200 kms in central and eastern regions. It is broader in western region.
There are also altitudinal variations across the regions. The Himalayas comprise
three parallel ranges. These ranges are separated with deep valleys and extensive
plateaus.
The northern most range is known as Greater Himalayas or Himadri. This range
is the most continuous consisting of the highest p

In [73]:
import os
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEndpoint
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()
# Import the new LCEL components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Load API keys
load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINFACE_ENDPOIN")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")

# 2. Initialize Embeddings & Connect to Pinecone Index
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-mpnet-base-v2"
)

vector_store = PineconeVectorStore(
    index_name="social-textbook-index", 
    embedding=embeddings
)

# 3. Create the Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile"
)

# 5. Define the Prompt Template
template = """You are an expert assistant for answering questions based on textbook material.
Use the following pieces of retrieved context to answer the question.
If you do not know the answer based on the context, say that you don't know.

Context:
{context}

Question: {input}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# 6. Helper function to format the retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7. Construct the LCEL Chain using the pipe operator (|)
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 8. Invoke the Chain
user_question = "The Himalayan ranges run in the west-east direction in the form of an arch with a distance of about _______"

# Because of StrOutputParser, the response is just a clean string
response = rag_chain.invoke(user_question)

print("--- Answer ---")
print(response)

--- Answer ---
2,400 kms.


In [76]:
rag_chain.invoke("what are the causes of world war")

'According to the context, one of the causes of World War is mentioned as "Aggressive Nationalism", which created pride in nations and hatred against neighboring countries. Additionally, the context mentions the ideology of nationalism as a positive impulse that led to the unification of Germany and Italy, but also created conflicts. \n\nHowever, it seems that the context does not provide a comprehensive list of causes of World War. It does mention the immediate cause of World War II, which was when the German Army entered Poland on September 1, 1939, but it does not provide a detailed list of causes for both World War I and World War II. \n\nIt is also mentioned that the western capitalist countries\' fear of communism and their policy of \'appeasement\' of Hitler contributed to the lead-up to World War II. But a complete list of causes is not provided in the given context.'